# Colocalization

In [ ]:
library(coloc)
library(data.table)

coloc version 5.2.3

In [ ]:
library(coloc)
library(data.table)


# Load GWAS data for the region

gwas <- fread("/path/to/gwas/output/saige/chr4_CATPD")
gwas <- gwas[POS > 89724099 & POS < 89838315 ]
gwas

In [ ]:
gwas$beta <- as.numeric(gwas$BETA)
gwas$se <- as.numeric(gwas$SE)
gwas$pvalue <- as.numeric(gwas$p.value)
gwas <- gwas[!is.na(beta) & !is.na(se) & se > 0]
gwas$varbeta <- gwas$se^2

# Variant ID generate
gwas$variant_id <- paste0(gwas$CHR, "_", gwas$POS, "_", gwas$Allele1, "_", gwas$Allele2, "_b38")
gwas$variant_id_flip <- paste0(gwas$CHR, "_", gwas$POS, "_", gwas$Allele2, "_", gwas$Allele1, "_b38")

cat("GWAS variants in region:", nrow(gwas), "\n")

In [ ]:
# Load GTEx signif pairs for each brain tissue


setwd('./coloc/GTEx_Analysis_v8_eQTL')
tissues <- c(
    "Brain_Substantia_nigra",
    "Brain_Caudate_basal_ganglia",
    "Brain_Putamen_basal_ganglia",
    "Brain_Cortex",
    "Brain_Frontal_Cortex_BA9",
    "Brain_Hippocampus",
    "Brain_Cerebellum",
    "Brain_Nucleus_accumbens_basal_ganglia",
    "Brain_Amygdala",
    "Brain_Anterior_cingulate_cortex_BA24",
    "Brain_Hypothalamus",
    "Brain_Spinal_cord_cervical_c-1",
    "Brain_Cerebellar_Hemisphere",
    "Whole_Blood"
)

# GTEx sample sizes (put the exact numbers from GTEx v8)
tissue_n <- list(
    Brain_Substantia_nigra = 114,
    Brain_Caudate_basal_ganglia = 194,
    Brain_Putamen_basal_ganglia = 170,
    Brain_Cortex = 205,
    Brain_Frontal_Cortex_BA9 = 175,
    Brain_Hippocampus = 165,
    Brain_Cerebellum = 209,
    Brain_Nucleus_accumbens_basal_ganglia = 202,
    Brain_Amygdala = 129,
    Brain_Anterior_cingulate_cortex_BA24 = 147,
    Brain_Hypothalamus = 170,
    `Brain_Spinal_cord_cervical_c-1` = 126,
    Brain_Cerebellar_Hemisphere = 175,
    Whole_Blood = 670
)

In [ ]:
results_list <- list()

In [ ]:
for (tissue in tissues) {
    fname <- paste0(tissue, ".v8.signif_variant_gene_pairs.txt.gz")
    if (!file.exists(fname)) {
        cat("Skipping", tissue, "- file not found\n")
        next
    }
    
    cat("\nProcessing:", tissue, "\n")
    
    eqtl_all <- fread(fname)
    
    # Filter to region
    eqtl_all$pos <- as.numeric(sapply(strsplit(eqtl_all$variant_id, "_"), `[`, 2))
    eqtl_all$chr <- sapply(strsplit(eqtl_all$variant_id, "_"), `[`, 1)
    eqtl_region <- eqtl_all[chr == "chr4" & pos > 89724099 & pos < 89838315]
    
    if (nrow(eqtl_region) == 0) {
        cat("  No significant eQTLs in region\n")
        next
    }
    
    cat("  Found", nrow(eqtl_region), "significant eQTLs in region\n")
    cat("  Genes:", paste(unique(eqtl_region$gene_id), collapse=", "), "\n")
    
    # Run COLOC per gene
    for (gene in unique(eqtl_region$gene_id)) {
        eqtl_gene <- eqtl_region[gene_id == gene]
        
        # Match variants: try direct match, then flipped
        matched <- merge(gwas, eqtl_gene, by = "variant_id", all = FALSE)
        
        if (nrow(matched) == 0) {
            # Try flipped alleles
            matched <- merge(gwas, eqtl_gene, 
                           by.x = "variant_id_flip", by.y = "variant_id", all = FALSE)
            if (nrow(matched) > 0) {
                matched$slope <- -matched$slope  # flip effect direction
            }
        }
        
        if (nrow(matched) < 3) {
            cat("  Gene", gene, ": too few matched variants (", nrow(matched), ")\n")
            next
        }
        
        cat("  Gene", gene, ": matched", nrow(matched), "variants\n")
        
        n_eqtl <- tissue_n[[tissue]]
        if (is.null(n_eqtl)) n_eqtl <- 150  # fallback
        
        # Run COLOC
        tryCatch({
            res <- coloc.abf(
                dataset1 = list(
                    beta = matched$beta,
                    varbeta = matched$varbeta,
                    N = 6270,
                    type = "cc",
                    s = 0.5  
                ),
                dataset2 = list(
                    beta = matched$slope,
                    varbeta = matched$slope_se^2,
                    N = n_eqtl,
                    type = "quant"
                )
            )
            
            cat("    PP.H0=", round(res$summary["PP.H0.abf"], 3),
                " PP.H1=", round(res$summary["PP.H1.abf"], 3),
                " PP.H2=", round(res$summary["PP.H2.abf"], 3),
                " PP.H3=", round(res$summary["PP.H3.abf"], 3),
                " PP.H4=", round(res$summary["PP.H4.abf"], 3), "\n")
            
            results_list[[paste(tissue, gene, sep="__")]] <- data.frame(
                tissue = tissue,
                gene = gene,
                n_variants = nrow(matched),
                PP.H0 = res$summary["PP.H0.abf"],
                PP.H1 = res$summary["PP.H1.abf"],
                PP.H2 = res$summary["PP.H2.abf"],
                PP.H3 = res$summary["PP.H3.abf"],
                PP.H4 = res$summary["PP.H4.abf"]
            )
        }, error = function(e) {
            cat("    COLOC failed:", e$message, "\n")
        })
    }
}

In [ ]:
# Summary

if (length(results_list) > 0) {
    results_df <- do.call(rbind, results_list)
    results_df <- results_df[order(-results_df$PP.H4), ]
    
    cat("\n", paste(rep("=", 60), collapse=""), "\n")
    cat("COLOC RESULTS SUMMARY\n")
    cat(paste(rep("=", 60), collapse=""), "\n\n")
    print(results_df)
    
    write.csv(results_df, "cd coloc_results.csv", row.names=FALSE)
    cat("\nSaved to coloc_results.csv\n")
    
    if (any(results_df$PP.H4 > 0.5)) {
        cat("\n*** COLOCALIZATION DETECTED (PP.H4 > 0.5) ***\n")
        print(results_df[results_df$PP.H4 > 0.5, ])
    }
} else {
    cat("No COLOC results\n")
}